# Hosting Memory-Enabled Strands Agents with OpenAI models in Amazon Bedrock AgentCore Runtime

In [26]:
import boto3
from botocore.exceptions import ClientError
import json
import time

sts_client = boto3.client('sts')
account_id = sts_client.get_caller_identity()['Account']
region = boto3.session.Session().region_name or 'us-east-1'

ecr_client = boto3.client('ecr', region_name = region)
iam_client = boto3.client('iam')
agent_core_client = boto3.client('bedrock-agentcore-control', region_name=region)

Let's set the names of the resources we will create and import the OpenAI API key 

In [16]:
from dotenv import load_dotenv
import os

ecr_repo_name = 'voice-agent-openai'
agent_name = 'voice-agent-openai'
cleaned_agent_name = agent_name.replace('-', '_')
iam_role_name = 'VoiceAgentOpenAIRole'

load_dotenv()
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')

## Store the Agent Image in ECR

Bedrock AgentCore runtimes requires an ECR stored image for the agent implementation, we will begin by creating the ECR repository

In [ ]:
try:
    response = ecr_client.create_repository(repositoryName=ecr_repo_name)
    print(response['repository']['repositoryUri'])
except ClientError as e:
    if e.response['Error']['Code'] == 'RepositoryAlreadyExistsException':
        print("   Repository already exists")
    else:
        raise

Then we login ECR and we push the agent image. Notice it could take up to 6 mins the first time an image is pushed.

In [ ]:
ecr_uri = f"{account_id}.dkr.ecr.{region}.amazonaws.com"
image_uri = f"{ecr_uri}/{ecr_repo_name}:latest"

# Login to ECR
login_password = ecr_client.get_authorization_token()['authorizationData'][0]['authorizationToken']
!echo {login_password} | base64 -d | cut -d: -f2 | docker login --username AWS --password-stdin {ecr_uri}

# Build and push (run from agent folder)
!docker build -t {ecr_repo_name}:latest .
!docker tag {ecr_repo_name}:latest {image_uri}
!docker push {image_uri}

print(f"\nImage pushed to: {image_uri}")

Now that the agent container has been pushed and stored in ECR we can proceed creating the Agent.

## Create the Agent

As a first step we create the IAM role and Trust Policy for the Agent to be able to run. The definitions are contained in the files [agent_role.json](./agent_role.json) and [agent_trust_policy.json](./agent_trust_policy.json).

In [ ]:
# Check if role already exists
try:
    response = iam_client.get_role(RoleName=iam_role_name)
    role_arn = response['Role']['Arn']
    print(f"ℹ️  IAM role {iam_role_name} already exists")
    print(f"✅ Using existing role: {role_arn}")
except ClientError as e:
    if e.response['Error']['Code'] == 'NoSuchEntity':
        print(f"Creating IAM role: {iam_role_name}...")
        
        # Load policy files
        with open('agent_trust_policy.json', 'r') as f:
            trust_policy = json.load(f)
        
        with open('agent_role.json', 'r') as f:
            agent_role_policy = f.read().replace('${ACCOUNT_ID}', account_id)
        
        # Create the IAM role
        response = iam_client.create_role(
            RoleName=iam_role_name,
            AssumeRolePolicyDocument=json.dumps(trust_policy)
        )
        role_arn = response['Role']['Arn']
        print(f"✅ Role created: {role_arn}")
        
        # Attach the inline policy
        iam_client.put_role_policy(
            RoleName=iam_role_name,
            PolicyName=f"{iam_role_name}Policy",
            PolicyDocument=agent_role_policy
        )
        print("✅ Policy attached. Waiting 30 secs for role propagation...")
        time.sleep(30)
        print("✅ Done")
    else:
        raise

The next step is to deploy the AgentCore memory and store the memory ID, so that can be passed to AgentCore Runtime when creating the running Agent.

In [ ]:
try:
    memory = agent_core_client.create_memory(
        name=f'{cleaned_agent_name}_memory',
        description='Memory for voice agent based on OpenAI',
        eventExpiryDuration=7
    )
except agent_core_client.exceptions.ValidationException as e:
    if 'already exists' in str(e):
        memories = agent_core_client.list_memories()
        memory_obj = next(
            m for m in memories['memories']
            if m['id'].startswith(f'{cleaned_agent_name}_memory')
        )
        memory = {'memory': memory_obj}
    else:
        raise

memory_id = memory['memory']['id']
print(memory_id)

Now that the memory has been created we can create and deploy the Agent Runtime. There will be some cases in which you want to update the Agent Runtime as you are i.e. modifying the agent definition, and you want to iterate over the runtime versions. Select the right case from below

### Create the Agent Runtime

In [ ]:
agent_runtime_name = f'{cleaned_agent_name }_runtime'
agent_runtime = agent_core_client.create_agent_runtime(
    agentRuntimeName = agent_runtime_name,
    agentRuntimeArtifact = {'containerConfiguration':{'containerUri':image_uri}},
    networkConfiguration = {'networkMode':'PUBLIC'},
    roleArn = role_arn,
    requestHeaderConfiguration = {
        'requestHeaderAllowlist': [
            'X-Amzn-Bedrock-AgentCore-Runtime-Custom-VoiceId'
        ]
    },
    environmentVariables = {
            'MEMORY_ID': memory_id,
            'OPENAI_API_KEY': OPENAI_API_KEY
            }
)
agent_runtime_id = agent_runtime['agentRuntimeId']
print("✅ Agent Runtime created:")
print(agent_runtime['agentRuntimeId'])
print(agent_runtime['agentRuntimeVersion'])
print(agent_runtime['agentRuntimeArn'])


### Update the Agent Runtime

In [ ]:
agent_runtime = agent_core_client.update_agent_runtime(
    agentRuntimeId = agent_runtime_id,
    agentRuntimeArtifact = {'containerConfiguration':{'containerUri':image_uri}},
    networkConfiguration = {'networkMode':'PUBLIC'},
    roleArn = role_arn,
    requestHeaderConfiguration = {
        'requestHeaderAllowlist': [
            'X-Amzn-Bedrock-AgentCore-Runtime-Custom-VoiceId'
        ]
    },
    environmentVariables = {
            'MEMORY_ID': memory_id,
            'OPENAI_API_KEY': OPENAI_API_KEY
            }
)
print("✅ Agent Runtime updated:")
print(agent_runtime['agentRuntimeId'])
print(agent_runtime['agentRuntimeVersion'])
print(agent_runtime['agentRuntimeArn'])

Now the agent is ready to serve requests make note of the Agent Runtime ARN, as it will be needed for the next steps. Refer to the [README.md](../README.md) for the next steps on how to run a local WebSocket client to interact with the newly created agent.

## Cleanup

Delete AgentCore Runtime

In [ ]:
try:
    agent_core_client.delete_agent_runtime(agentRuntimeId=agent_runtime_id)
    print("✅ Agent runtime deleted")
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceNotFoundException':
        print("ℹ️  Agent runtime not found, skipping")
    else:
        raise


Delete AgentCore Memory

In [ ]:
try:
    agent_core_client.delete_memory(memoryId=memory_id)
    print("✅ Memory deleted")
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceNotFoundException':
        print("ℹ️  Memory not found, skipping")
    else:
        raise

Delete IAM Role and Policy

In [ ]:
try:
    iam_client.delete_role_policy(RoleName=iam_role_name, PolicyName=f"{iam_role_name}Policy")
    iam_client.delete_role(RoleName=iam_role_name)
    print("✅ IAM role and policy deleted")
except ClientError as e:
    if e.response['Error']['Code'] == 'NoSuchEntity':
        print("ℹ️  IAM role not found, skipping")
    else:
        raise

Delete ECR Repository

In [ ]:
try:
    ecr_client.delete_repository(repositoryName=ecr_repo_name, force=True)
    print("✅ ECR repository and images deleted")
except ClientError as e:
    if e.response['Error']['Code'] == 'RepositoryNotFoundException':
        print("ℹ️  ECR repository not found, skipping")
    else:
        raise
